In [1]:
import pandas as pd
import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Cargar datos
df = pd.read_csv("ai4i2020.csv")
features = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

# 2. Guardar valores NORMALES (Esto es para la PRESCRIPCIÓN)
# Si algo falla, el sistema recomendará volver a estos valores medianos.
normal_op = df[df['Machine failure'] == 0][features].median().to_dict()
joblib.dump(normal_op, "normal_op.pkl")

# 3. Entrenamiento con SMOTE para balancear fallas
X = df[features]
y = df["Machine failure"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
joblib.dump(pipeline, "industry_model.pkl")
print(" Celda 1: Inteligencia de fallas lista.")

✅ Celda 1: Inteligencia de fallas lista.


In [2]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import numpy as np

# ---------------------------
# Inicializar API
# ---------------------------
app = FastAPI(
    title="Industrial Machine Monitoring API",
    description="Sistema de monitoreo con predicción y prescripción de fallas en maquinaria industrial",
    version="1.0"
)

# ---------------------------
# Cargar modelo y valores ideales
# ---------------------------
model = joblib.load("industry_model.pkl")
ideals = joblib.load("normal_op.pkl")

# ---------------------------
# Modelo de entrada
# ---------------------------
class MachineInput(BaseModel):

    air_temp: float = Field(..., example=298.5)
    process_temp: float = Field(..., example=308.5)
    rpm: int = Field(..., example=1500)
    torque: float = Field(..., example=45.0)
    tool_wear: int = Field(..., example=120)


# ---------------------------
# Endpoint de prueba
# ---------------------------
@app.get("/")
def home():
    return {
        "message": "API de Monitoreo Industrial activa",
        "modelo": "Random Forest Predictive Maintenance"
    }


# ---------------------------
# Predicción + Prescripción
# ---------------------------
@app.post("/predict")
def predict_and_prescribe(data: MachineInput):

    # Convertir datos a formato modelo
    X = np.array([
        [
            data.air_temp,
            data.process_temp,
            data.rpm,
            data.torque,
            data.tool_wear
        ]
    ])

    try:
        # Probabilidad de falla
        prob = model.predict_proba(X)[0][1]
    except Exception as e:
        return {
            "error": f"No se pudo calcular la probabilidad: {str(e)}",
            "detalle": "Verifica que industry_model.pkl esté entrenado y accesible."
        }

    # Clasificación
    prediction = 1 if prob > 0.5 else 0

    # ---------------------------
    # Nivel de riesgo
    # ---------------------------
    if prob < 0.3:
        risk = "BAJO"
    elif prob < 0.6:
        risk = "MEDIO"
    else:
        risk = "ALTO"

    # ---------------------------
    # Motor Prescriptivo
    # ---------------------------
    action = "Mantener parámetros actuales."

    if prediction == 1:

        recs = []

        if data.torque > ideals["Torque [Nm]"]:
            recs.append(f"Reducir torque a {round(ideals['Torque [Nm]'],2)} Nm")

        if data.tool_wear > 180:
            recs.append("Cambio de herramienta urgente")

        if data.process_temp > ideals["Process temperature [K]"]:
            recs.append(f"Reducir temperatura de proceso a {round(ideals['Process temperature [K]'],2)} K")

        if recs:
            action = " | ".join(recs)
        else:
            action = "Revisión técnica recomendada."

    # ---------------------------
    # Respuesta API
    # ---------------------------
    return {
        "probabilidad_falla": round(prob, 3),
        "probabilidad_porcentaje": f"{round(prob*100,1)}%",
        "estado_maquina": "CRÍTICO" if prediction == 1 else "OPERANDO NORMAL",
        "nivel_riesgo": risk,
        "accion_prescriptiva": action
    }


C:\Users\NewUser\AppData\Local\Temp\ipykernel_10112\3735376666.py:26: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  air_temp: float = Field(..., example=298.5)
C:\Users\NewUser\AppData\Local\Temp\ipykernel_10112\3735376666.py:27: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  process_temp: float = Field(..., example=308.5)
C:\Users\NewUser\AppData\Local\Temp\ipykernel_10112\3735376666.py:28: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json

In [4]:
#uvicorn main:app --reload # para anaconda pronmt

In [10]:
import requests
import time
import random
from datetime import datetime

def simular_monitoreo(intervalo=3):

    url = "http://127.0.0.1:8000/predict"

    print(" Sistema de monitoreo industrial iniciado")
    print("Presiona STOP en Jupyter para detener\n")

    try:

        while True:

            # -----------------------------
            # Simulación de sensores
            # -----------------------------
            payload = {
                "air_temp": round(random.uniform(295, 305), 1),
                "process_temp": round(random.uniform(305, 315), 1),
                "rpm": random.randint(1300, 1600),
                "torque": round(random.uniform(35, 45), 1),
                "tool_wear": random.randint(0, 200)
            }

            # -----------------------------
            # Simulación de anomalía
            # -----------------------------
            if random.random() < 0.1:
                payload["torque"] = round(random.uniform(65, 80), 1)
                payload["tool_wear"] = random.randint(200, 240)
                print("\n ANOMALÍA DETECTADA EN SENSORES")

            # -----------------------------
            # Llamada a la API
            # -----------------------------
            try:
                response = requests.post(url, json=payload)
                data = response.json()

                # -----------------------------
                # Monitor en consola
                # -----------------------------
                print("\n------------------------------------")
                print("🕒 Hora:", datetime.now().strftime("%H:%M:%S"))

                print("\n Datos de sensores")
                print("Air Temp:", payload["air_temp"], "K")
                print("Process Temp:", payload["process_temp"], "K")
                print("RPM:", payload["rpm"])
                print("Torque:", payload["torque"], "Nm")
                print("Tool Wear:", payload["tool_wear"], "min")

                print("\n Resultado del modelo")
                print("Respuesta completa del modelo:", data)

                # Uso de .get() para evitar KeyError si la clave no existe
                print("Probabilidad de falla:", data.get("probabilidad_falla", "No disponible"))
                print("Estado máquina:", data.get("estado", "No disponible"))
                print("Nivel de riesgo:", data.get("nivel_riesgo", "No disponible"))

                print("\n Acción recomendada")
                print(data.get("accion_prescriptiva", "No disponible"))

                print("------------------------------------")

            except Exception as e:
                print(" Ocurrió un error:", type(e).__name__, "-", e)

            time.sleep(intervalo)

    except KeyboardInterrupt:
        print("\n🛑 Monitoreo detenido por el usuario.")


# Ejecutar simulación
simular_monitoreo()


 Sistema de monitoreo industrial iniciado
Presiona STOP en Jupyter para detener


------------------------------------
🕒 Hora: 18:46:03

 Datos de sensores
Air Temp: 296.5 K
Process Temp: 308.3 K
RPM: 1467
Torque: 44.6 Nm
Tool Wear: 91 min

 Resultado del modelo
Respuesta completa del modelo: {'probabilidad_falla': '0.0%', 'estado': 'OK', 'accion_prescriptiva': 'Mantener parámetros actuales.'}
Probabilidad de falla: 0.0%
Estado máquina: OK
Nivel de riesgo: No disponible

 Acción recomendada
Mantener parámetros actuales.
------------------------------------

------------------------------------
🕒 Hora: 18:46:06

 Datos de sensores
Air Temp: 295.9 K
Process Temp: 311.6 K
RPM: 1395
Torque: 38.8 Nm
Tool Wear: 127 min

 Resultado del modelo
Respuesta completa del modelo: {'probabilidad_falla': '0.0%', 'estado': 'OK', 'accion_prescriptiva': 'Mantener parámetros actuales.'}
Probabilidad de falla: 0.0%
Estado máquina: OK
Nivel de riesgo: No disponible

 Acción recomendada
Mantener parámetros a

In [6]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import random
import time

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

st.set_page_config(page_title="Industrial AI Monitoring", layout="wide")

st.title("🏭 Industrial AI Monitoring System")

# -----------------------------
# CARGAR DATASET
# -----------------------------

@st.cache_data
def load_data():

    df = pd.read_csv("ai4i2020.csv")

    le = LabelEncoder()
    df["Type"] = le.fit_transform(df["Type"])

    features = [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]

    X = df[features]
    y = df["Machine failure"]

    return df, X, y

df, X, y = load_data()

# -----------------------------
# ENTRENAR MODELO ML
# -----------------------------

@st.cache_resource
def train_model(X,y):

    X_train,X_test,y_train,y_test = train_test_split(
        X,y,test_size=0.2,random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42
    )

    model.fit(X_train,y_train)

    return model

model = train_model(X,y)

# -----------------------------
# ANOMALY DETECTION
# -----------------------------

@st.cache_resource
def train_anomaly_model(X):

    iso = IsolationForest(
        contamination=0.03,
        random_state=42
    )

    iso.fit(X)

    return iso

anomaly_model = train_anomaly_model(X)

# -----------------------------
# CONFIG DASHBOARD
# -----------------------------

st.sidebar.header("Configuración")

auto_refresh = st.sidebar.checkbox("Streaming sensores",True)
interval = st.sidebar.slider("Intervalo (seg)",1,10,3)

machines = [f"Machine {i}" for i in range(1,6)]

if "history" not in st.session_state:
    st.session_state.history = {m:[] for m in machines}

alerts = []

# -----------------------------
# STREAMING DE SENSORES
# -----------------------------

sensor_data = []

for machine in machines:

    air_temp = random.uniform(296,304)
    process_temp = random.uniform(306,314)
    rpm = random.randint(1400,1600)
    torque = random.uniform(30,70)
    tool_wear = random.randint(0,250)

    row = [
        air_temp,
        process_temp,
        rpm,
        torque,
        tool_wear
    ]

    pred_prob = model.predict_proba([row])[0][1]
    pred_class = model.predict([row])[0]

    anomaly = anomaly_model.predict([row])[0]

    rul = 250 - tool_wear

    # causas de falla
    TWF = int(tool_wear > 200)
    HDF = int(process_temp > 312)
    PWF = int(torque > 55)
    OSF = int(rpm > 1580)

    sensor_data.append({
        "Machine":machine,
        "Failure Prob":pred_prob*100,
        "RUL":rul,
        "TWF":TWF,
        "HDF":HDF,
        "PWF":PWF,
        "OSF":OSF,
        "Anomaly":anomaly
    })

    st.session_state.history[machine].append(pred_prob*100)

    st.session_state.history[machine] = st.session_state.history[machine][-30:]

    if pred_prob > 0.8:
        alerts.append(machine)

# -----------------------------
# ALERTAS
# -----------------------------

st.subheader("🚨 Alertas")

if alerts:

    for a in alerts:
        st.error(f"{a} riesgo crítico de falla")

else:

    st.success("Sistema estable")

# -----------------------------
# MONITOREO MAQUINAS
# -----------------------------

st.subheader("🏭 Estado de máquinas")

cols = st.columns(5)

for i,machine in enumerate(machines):

    with cols[i]:

        st.markdown(f"### {machine}")

        prob = sensor_data[i]["Failure Prob"]
        rul = sensor_data[i]["RUL"]

        st.metric("Prob falla",f"{prob:.1f}%")
        st.metric("RUL",f"{rul} min")

        if prob > 80:
            st.error("CRÍTICO")

        elif prob > 60:
            st.warning("RIESGO")

        else:
            st.success("NORMAL")

        df_hist = pd.DataFrame(
            st.session_state.history[machine],
            columns=["Prob"]
        )

        st.line_chart(df_hist)

# -----------------------------
# PANEL DE CAUSAS DE FALLA
# -----------------------------

st.subheader("📊 Causas de falla")

df_panel = pd.DataFrame(sensor_data)

cause_counts = df_panel[["TWF","HDF","PWF","OSF"]].sum()

st.bar_chart(cause_counts)

st.write("""
TWF → Tool Wear Failure  
HDF → Heat Dissipation Failure  
PWF → Power Failure  
OSF → Overstrain Failure
""")

# -----------------------------
# ANOMALY DETECTION
# -----------------------------

st.subheader("📉 Detección de anomalías")

anomaly_df = df_panel[df_panel["Anomaly"]==-1]

if len(anomaly_df)>0:

    st.error("Anomalías detectadas")

    st.dataframe(anomaly_df)

else:

    st.success("No hay anomalías")

# -----------------------------
# STREAMING
# -----------------------------

if auto_refresh:

    time.sleep(interval)
    st.rerun()

Overwriting dashboard.py


In [ ]:
#streamlit run dashboard.py# Ver el dashboard